# Claude Code + MCP  →  Cortex Agents + CoWork
## Notebook 01 — Shared Foundation (Part 0)

**Scenario:** a B2B sales-intelligence / go-to-market (GTM) SaaS company scores every
sales-rep email with AI and mines winning email patterns. Over four notebooks we migrate that workload
from an **external brain** (Claude Code over the Snowflake-managed MCP server) to an **in-data-plane
multi-agent architecture** (Cortex Agents + CoWork).

### The thesis — four pillars, all provable on the Snowflake side
The migration is not a data rebuild; it is **relocating the reasoning loop into the data plane**. The
payoff shows up across four pillars, each demonstrable directly with SQL:

1. **Governance & Security** — per-tool least-privilege grants, owner's-rights execution, and explicit
   behavioral rules bound to the agent.
2. **Cost Control** — a hard orchestration budget on the reasoning loop, a targeting gate that skips
   low-value work, and cheap-model-first routing.
3. **Observability** — server-side spans for every agent run, plus a native evaluation loop.
4. **Data Locality** — tool results never leave the Snowflake perimeter; no external connector to keep alive.

> **Why the Claude + MCP comparison is qualitative:** the external brain runs on Anthropic's side, so its
> planning latency and token cost live **outside** Snowflake. This demo therefore compares the two
> architectures on what Snowflake governs and measures in-plane — the four pillars above — and treats
> in-plane latency as an observable, server-side metric rather than a head-to-head speed number.

This notebook establishes the shared foundation **both** front doors reuse: the data, the semantic view
(Cortex Analyst), the Cortex Search service, and a governed SQL tool.

### What you'll do
1. Connect and tour the synthetic GTM email corpus.
2. Confirm the three reusable tools: Cortex Analyst (semantic view), Cortex Search, governed UDF.
3. Prove the Governance pillar's foundation: each tool is granted separately (least privilege).

**Prerequisite:** run `lab/setup.sql` once before this notebook.

---
## Section 1: Connect

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE GTMAGENTS").collect()
session.sql("USE SCHEMA GTMAGENTS.DEMO").collect()
session.sql("USE WAREHOUSE GTMAGENTS_WH").collect()
print("Connected to GTMAGENTS.DEMO")

---
## Section 2: Tour the data

Four tables: `REPS`, `EMAILS` (subject/body text generated across good / mixed / poor quality tiers),
`OUTCOMES` (deal progression + revenue), and `EMAIL_FRAMEWORK` (the best-practice rubric).
The latent quality tier drives engagement, so intent scoring (Part B) will be genuinely learnable from the text.

In [ ]:
# Row counts
session.sql("""
    SELECT 'REPS' AS tbl, COUNT(*) AS row_count FROM REPS
    UNION ALL SELECT 'EMAILS', COUNT(*) FROM EMAILS
    UNION ALL SELECT 'OUTCOMES', COUNT(*) FROM OUTCOMES
    UNION ALL SELECT 'EMAIL_FRAMEWORK', COUNT(*) FROM EMAIL_FRAMEWORK
""").show()

In [ ]:
# Quality spread — reply/win rates separate cleanly by tier (1=poor, 2=mixed, 3=good)
session.sql("""
    SELECT e.quality_tier,
           COUNT(*) AS emails,
           ROUND(AVG(IFF(e.replied,1,0)),3) AS reply_rate,
           ROUND(AVG(IFF(o.won,1,0)),3)     AS win_rate,
           SUM(o.revenue)                    AS revenue
    FROM EMAILS e JOIN OUTCOMES o ON o.email_id = e.id
    GROUP BY e.quality_tier ORDER BY e.quality_tier
""").show()

In [ ]:
# Sample one email per tier so you can see the text quality difference
session.sql("""
    SELECT quality_tier, subject, body FROM EMAILS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY quality_tier ORDER BY RANDOM()) = 1
    ORDER BY quality_tier
""").show(max_width=200)

---
## Section 3: Tool 1 — Cortex Analyst (semantic view)

`EMAIL_GTM_SV` is the governed semantic view over emails + reps + outcomes. Cortex Analyst turns a natural-
language question into a `SEMANTIC_VIEW(...)` query. Below is the SQL Analyst generates for
*"What is the reply rate and win rate by region?"* — the **same** semantic view is reused by the MCP server
(Part A) and the Recommendation agent (Part B), so business definitions never drift between the two worlds.

In [ ]:
session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        EMAIL_GTM_SV
        DIMENSIONS reps.region
        METRICS reply_rate, win_rate, total_revenue
    ) ORDER BY total_revenue DESC
""").show()

---
## Section 4: Tool 2 & 3 — Cortex Search + governed SQL tool

`FRAMEWORK_SEARCH` retrieves best-practice rubric text (used by the Coaching agent to rewrite weak emails).
`GTM_TEAM_PERFORMANCE(region)` is a **governed** read-only UDF that returns curated aggregate GTM metrics as a
single JSON cell — never raw rows — honoring the Cortex custom-tool contract and the least-privilege story.

In [ ]:
# Cortex Search preview — retrieve the rubric principles most relevant to a weak email
session.sql("""
    SELECT PARSE_JSON(
      SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'GTMAGENTS.DEMO.FRAMEWORK_SEARCH',
        '{"query": "email has no clear call to action and is generic", "limit": 3}'
      ))['results'] AS results
""").show(max_width=200)

In [ ]:
# Governed SQL tool — aggregate team performance for one region (returns a JSON array, single cell)
session.sql("SELECT GTM_TEAM_PERFORMANCE('AMER') AS amer_performance").show(max_width=200)

In [ ]:
# Governance pillar (foundation): access to a tool is granted per object — least privilege.
# Both front doors (MCP server, and the AFTER supervisor) will consume these SAME governed objects,
# so the access surface is defined once, here, on the objects themselves.
for obj_type, obj in [
    ('VIEW',      'GTMAGENTS.DEMO.EMAIL_GTM_SV'),        # Cortex Analyst semantic view
    ('FUNCTION',  'GTMAGENTS.DEMO.GTM_TEAM_PERFORMANCE(VARCHAR)'),  # governed UDF
]:
    print(f"\nGrants on {obj_type} {obj}:")
    session.sql(f"SHOW GRANTS ON {obj_type} {obj}").select('"privilege"', '"granted_to"', '"grantee_name"').show(max_width=200)

# Cortex Search is a service object — list its grants the same way.
print("\nGrants on CORTEX SEARCH SERVICE GTMAGENTS.DEMO.FRAMEWORK_SEARCH:")
session.sql("SHOW GRANTS ON CORTEX SEARCH SERVICE GTMAGENTS.DEMO.FRAMEWORK_SEARCH").select('"privilege"', '"granted_to"', '"grantee_name"').show(max_width=200)

---
## Summary

The shared foundation is live: synthetic GTM email data plus three governed tools
(Cortex Analyst semantic view, Cortex Search, governed UDF), each granted **per object** to
`GTMAGENTS_ROLE` — the least-privilege base of the **Governance** pillar. Both front doors reuse these
same objects, so business definitions and the access surface never drift.

Next, **`gtm-02-before-mcp`** exposes these tools through a Snowflake-managed MCP server so Claude Code
can drive them as an external brain — and shows the **control gap** that opens up once the reasoning loop
lives outside the data plane.